This notebook contains the generation of synthetic data based on the real Beijing dataset. Created utilizing the SDV library! https://datacebo.com/sdv-dev/

In [1]:
# load the real data as a base + imports

import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from pathlib import Path
import re
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import Metadata

df = pd.read_csv(
    "data/real_pruned/clip_alpha_0.000.csv",
    parse_dates=["timestamp"]
)

df = df.sort_values("timestamp").reset_index(drop=True)

In [2]:

data = df
metadata = Metadata.detect_from_dataframe(data)
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(data)
synthetic_data = synthesizer.sample(num_rows=1000)

c:\Users\lenni\Documents\Thesis\.venv\Lib\site-packages\sdv\single_table\base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [3]:
from sdv.evaluation.single_table import run_diagnostic

diagnostic = run_diagnostic(
    real_data=data,
    synthetic_data=synthetic_data,
    metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 9/9 [00:00<00:00, 527.65it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 850.08it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



In [4]:
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    real_data=data,
    synthetic_data=synthetic_data,
    metadata=metadata
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 327.96it/s]|
Column Shapes Score: 74.87%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 299.73it/s]|
Column Pair Trends Score: 99.48%

Overall Score (Average): 87.18%



In [ ]:
from sdv.evaluation.single_table import get_column_plot

fig = get_column_plot(
    real_data=data,
    synthetic_data=synthetic_data,
    column_name='TEMP',
    metadata=metadata
)

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [7]:
from sdv.evaluation.single_table import get_column_pair_plot

fig = get_column_pair_plot(
    real_data=data,
    synthetic_data=synthetic_data,
    column_names=['TEMP', 'pm2.5'],
    metadata=metadata
)

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [12]:
# save synthetic dataset
alpha = 0.000
out_dir = Path("data/synthetic_pruned")
fname = out_dir / f"clip_alpha_{alpha:.3f}.csv"
out_dir.mkdir(exist_ok=True)
synthetic_data.to_csv(fname, index=False)